In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the dataset
dataset=pd.read_csv("C:/Users/Sinduja/Documents/AI_sinduja/Hope_AI/MLDSCapstoneProject/ThyroidCancerRecurrencePredictionProject/DataPreprocessing/cleaned_dataset.csv",index_col=None)
dataset=pd.get_dummies(dataset,drop_first=True)
X=dataset[['Age', 'Risk_Intermediate', 'Risk_Low', 'N_N1b', 'Response_Excellent',
       'Response_Structural Incomplete']]
y=dataset['Recurred_Yes']

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale features (essential for Logistic Regression convergence)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Define the hyperparameter grid
# C: Inverse of regularization strength (smaller values = stronger regularization)
# solver: Algorithm to use for optimization
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],  # 'l1' requires specific solvers like 'liblinear' or 'saga'
    'solver': ['lbfgs', 'liblinear']
}

# 5. Initialize the model and GridSearchCV
log_reg = LogisticRegression(max_iter=5000)
grid_search = GridSearchCV(
    estimator=log_reg, 
    param_grid=param_grid, 
    cv=5,            # 5-fold cross-validation
    scoring='accuracy', 
    verbose=1
)

# 6. Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# 7. Print results
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Training Accuracy: {grid_search.best_score_:.4f}")

# 8. Evaluate on unseen test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
#print(y_pred)

print("\n--- Test Set Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

import pickle
filename='C:/Users/Sinduja/Documents/AI_sinduja/Hope_AI/MLDSCapstoneProject/ThyroidCancerRecurrencePredictionProject/FinalModel/TCRpredictionModel.sav'
pickle.dump(best_model,open(filename,'wb'))
sc_file='C:/Users/Sinduja/Documents/AI_sinduja/Hope_AI/MLDSCapstoneProject/ThyroidCancerRecurrencePredictionProject/FinalModel/Scaler.sav'
pickle.dump(scaler,open(sc_file,'wb'))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Hyperparameters: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Best Training Accuracy: 0.9444

--- Test Set Performance ---
Accuracy Score: 0.9740
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        58
           1       1.00      0.89      0.94        19

    accuracy                           0.97        77
   macro avg       0.98      0.95      0.96        77
weighted avg       0.97      0.97      0.97        77

